# Notebook 1: scRNA-seq Data Loading, QC, and Initial Analysis

This notebook demonstrates foundational scRNA-seq analysis using Python and Scanpy.

**Topics covered:**
- Loading count matrices
- Basic quality control (QC)
- Filtering cells and genes
- Normalization
- Initial visualization

## Installation and Setup

Required libraries:
```bash
pip install scanpy pandas numpy matplotlib seaborn anndata
```

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting settings
sns.set_style('whitegrid')
sc.settings.verbosity = 3  # Increase verbosity
sc.settings.logfile = 'scanpy_analysis.log'

# Show versions
print(f'Scanpy version: {sc.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## Part 1: Loading Data

We'll work with a toy dataset. In real experiments, you would load your own count matrix from 10x, CEL-seq, SMART-seq, etc.

In [ ]:
# Generate a toy dataset for demonstration
# In practice, you would load your own data:
# adata = sc.read_h5ad('filtered_feature_bc_matrix.h5ad')  # 10x format
# adata = sc.read_csv('counts.csv', first_column_names=True)  # CSV format

# For this notebook, create synthetic data
np.random.seed(42)
n_obs = 1000  # Number of cells
n_vars = 5000  # Number of genes

# Create synthetic count matrix (sparse would be more realistic)
# Use negative binomial distribution to simulate realistic RNA-seq counts
counts = np.random.negative_binomial(5, 0.3, size=(n_obs, n_vars)).astype('float32')

# Create AnnData object (standard format for scRNA-seq)
adata = sc.AnnData(X=counts)

# Add metadata
adata.obs_names = [f'Cell_{i}' for i in range(n_obs)]
adata.var_names = [f'Gene_{i}' for i in range(n_vars)]
adata.var['gene_id'] = [f'ENSG{i:05d}' for i in range(n_vars)]

# Simulate cell type labels (in reality, these are discovered via clustering)
cell_types = np.random.choice(['T_cell', 'B_cell', 'Macrophage', 'Fibroblast'], n_obs, p=[0.3, 0.2, 0.25, 0.25])
adata.obs['cell_type'] = cell_types

print(adata)
print(f'\nCell type distribution:')
print(adata.obs['cell_type'].value_counts())

## Part 2: Quality Control

Assess basic QC metrics and identify problematic cells.

In [ ]:
# Calculate QC metrics
# This includes: total counts per cell, number of genes per cell, etc.
sc.pp.calculate_qc_metrics(adata, inplace=True)

print("QC metrics calculated. Available columns:")
print(adata.obs.columns.tolist())
print(f"\nFirst few rows of QC metrics:")
print(adata.obs[['n_counts', 'n_genes_by_counts']].head())

In [ ]:
# Visualize QC metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Total counts per cell (library size)
axes[0].hist(adata.obs['n_counts'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Total UMI counts per cell')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Library Size Distribution')
axes[0].axvline(adata.obs['n_counts'].median(), color='red', linestyle='--', label=f"Median: {adata.obs['n_counts'].median():.0f}")
axes[0].legend()

# Plot 2: Genes per cell
axes[1].hist(adata.obs['n_genes_by_counts'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of genes per cell')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Gene Detection per Cell')
axes[1].axvline(adata.obs['n_genes_by_counts'].median(), color='red', linestyle='--', label=f"Median: {adata.obs['n_genes_by_counts'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"QC Summary Statistics:")
print(adata.obs[['n_counts', 'n_genes_by_counts']].describe())

## Part 3: Cell Filtering

Remove low-quality cells based on QC metrics.

In [ ]:
# Define filtering thresholds
min_counts = 500  # Minimum UMI counts per cell
min_genes = 200   # Minimum genes per cell
max_genes = 5000  # Maximum genes (detect doublets/multiplets)

print(f"Initial dataset: {adata.n_obs} cells, {adata.n_vars} genes")
print(f"\nFiltering criteria:")
print(f"  - min_counts: {min_counts}")
print(f"  - min_genes: {min_genes}")
print(f"  - max_genes: {max_genes}")

# Apply filtering
sc.pp.filter_cells(adata, min_counts=min_counts)
print(f"\nAfter min_counts filter: {adata.n_obs} cells")

sc.pp.filter_cells(adata, min_genes=min_genes)
print(f"After min_genes filter: {adata.n_obs} cells")

adata = adata[adata.obs['n_genes_by_counts'] < max_genes, :]
print(f"After max_genes filter: {adata.n_obs} cells")

# Also filter genes: keep genes expressed in at least a few cells
sc.pp.filter_genes(adata, min_cells=3)
print(f"\nAfter gene filtering (min_cells=3): {adata.n_vars} genes")

## Part 4: Normalization

Normalize library sizes and log-transform.

In [ ]:
# Make a backup of raw counts for later use
adata.layers['counts'] = adata.X.copy()

# Normalize to 10,000 counts per cell (similar to CPM but scaled to 1e4)
sc.pp.normalize_total(adata, target_sum=1e4)
print("Normalized to 10,000 counts per cell")

# Log transform
sc.pp.log1p(adata)
print("Applied log(1+x) transformation")

# Store log-normalized data
adata.layers['log_normalized'] = adata.X.copy()

print(f"\nAfter normalization, X shape: {adata.X.shape}")
print(f"X range: [{np.min(adata.X):.2f}, {np.max(adata.X):.2f}]")

## Part 5: Gene Selection (Highly Variable Genes)

Identify genes with high biological variability for downstream analysis.

In [ ]:
# Select highly variable genes (HVGs)
# These genes show high variance across cells (likely informative for cell types)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')

print(f"Highly variable genes: {np.sum(adata.var['highly_variable'])} genes selected")
print(f"\nTop 10 HVGs by variance:")
hvg_df = adata.var[adata.var['highly_variable']].sort_values('dispersions_norm', ascending=False).head(10)
print(hvg_df[['mean', 'dispersions', 'dispersions_norm']])

In [ ]:
# Visualize mean vs. variance relationship
hvg_df_all = adata.var.sort_values('dispersions_norm', ascending=False)

plt.figure(figsize=(8, 6))
plt.scatter(np.log10(hvg_df_all['mean'] + 1e-4), hvg_df_all['dispersions'], 
           c=hvg_df_all['highly_variable'], cmap='viridis', alpha=0.6, s=20)
plt.xlabel('Mean expression (log10)')
plt.ylabel('Dispersion')
plt.title('Gene Expression Variability')
plt.colorbar(label='Highly Variable')
plt.show()

## Part 6: Dimensionality Reduction

Reduce high-dimensional data to 2D for visualization.

In [ ]:
# Scale data (zero-mean, unit variance)
sc.pp.scale(adata, max_value=10)
print("Data scaled to zero mean and unit variance")

# PCA
sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
print(f"PCA computed with 50 components")

# UMAP (requires leiden clustering first for some use cases, but we'll just do UMAP)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
print(f"Neighborhood graph computed")

sc.tl.umap(adata)
print(f"UMAP computed")

## Part 7: Visualization

Visualize cells in 2D reduced space.

In [ ]:
# Plot UMAP colored by cell type (known labels)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cell type
sc.pl.umap(adata, color='cell_type', ax=axes[0], title='Cells colored by type')

# Plot 2: Total counts (QC metric)
sc.pl.umap(adata, color='n_counts', ax=axes[1], title='Cells colored by library size')

plt.tight_layout()
plt.show()

print("UMAP visualization complete")

## Part 8: Gene Expression Visualization

Visualize specific genes across cells.

In [ ]:
# Select a few random genes to visualize
genes_to_plot = np.random.choice(adata.var_names, 4, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, gene in enumerate(genes_to_plot):
    sc.pl.umap(adata, color=gene, ax=axes[idx], title=f'Expression: {gene}')

plt.tight_layout()
plt.show()

## Part 9: Clustering

Identify cell clusters (communities in the neighbor graph).

In [ ]:
# Leiden clustering on neighborhood graph
sc.tl.leiden(adata, resolution=0.5)

print(f"Leiden clustering results:")
print(adata.obs['leiden'].value_counts().sort_index())

# Plot clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data')
plt.title('Leiden clusters')
plt.show()

## Part 10: Save Results

Save processed data for downstream analysis.

In [ ]:
# Save as HDF5
# adata.write_h5ad('processed_adata.h5ad')
# print("Data saved to processed_adata.h5ad")

# For now, just print summary
print(f"\n=== ANALYSIS SUMMARY ===")
print(f"Cells (after QC): {adata.n_obs}")
print(f"Genes (after filtering): {adata.n_vars}")
print(f"Highly variable genes: {np.sum(adata.var['highly_variable'])}")
print(f"Leiden clusters: {len(adata.obs['leiden'].unique())}")
print(f"\nData structure: {adata}")

## Summary

This notebook covered the essential steps of scRNA-seq analysis:

1. **Data loading**: Importing count matrices
2. **QC metrics**: Calculating library size, gene counts
3. **Filtering**: Removing low-quality cells and genes
4. **Normalization**: Library size and log transformation
5. **Gene selection**: Identifying highly variable genes
6. **Dimensionality reduction**: PCA and UMAP
7. **Visualization**: Exploring data in reduced space
8. **Clustering**: Identifying cell communities

**Next steps:** Differential expression analysis, cell-type annotation, trajectory inference (see later chapters).